# Digital Banking A/B Testing: Optimizing Loan Application Conversion

This notebook walks through data validation, experiment balance, primary conversion analysis, statistical testing, confidence intervals, secondary metrics, and exploratory segments.

**The dataset is synthetic.**


In [1]:
import sys, os

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))

In [2]:
from pathlib import Path
import pandas as pd
from src.data_preparation import load_data, quality_report, validate_data
from src.experiment_analysis import group_summary, experiment_result, segment_conversion

In [3]:
# Load data
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'/'digital_banking_ab_testing_synthetic_dataset.csv'
df=load_data(DATA)
df.head()

,user_id,experiment_group,age,gender,income,employment_status,credit_score,loan_amount,loan_term_months,application_started,application_completed,application_time_minutes,application_date,device_type,previous_customer,loan_approved
0,U000001,control,33,Female,53800,Self-employed,590,8000,24,1,0,NaN,2026-08-19,Web,1,0
1,U000002,treatment,37,Male,77800,Employed,697,21000,36,1,0,NaN,2026-08-20,Mobile,1,0
2,U000003,treatment,49,Male,64700,Employed,677,14900,48,1,0,NaN,2026-08-30,Mobile,1,0
3,U000004,control,29,Male,20400,Employed,653,2000,36,1,0,NaN,2026-08-23,Mobile,1,0
4,U000005,control,41,Female,34400,Unemployed,682,3700,24,1,0,NaN,2026-08-22,Web,0,0


## 1. Data understanding
Inspect shape, types, missing values, duplicates, and experiment groups before analyzing outcomes.


In [4]:
print('Shape:', df.shape)
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).head(10))
print('\nGroup counts:')
display(df['experiment_group'].value_counts())
print('\nSummary:')
display(df.describe(include='all').T)


Shape: (20000, 16)

Missing values:


application_time_minutes    16824
user_id                         0
experiment_group                0
age                             0
gender                          0
income                          0
employment_status               0
credit_score                    0
loan_amount                     0
loan_term_months                0
dtype: int64


Group counts:


experiment_group
control      10046
treatment     9954
Name: count, dtype: int64


Summary:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,20000,20000,U000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experiment_group,20000,2,control,10046,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,20000.0,NaN,NaN,NaN,36.425,21.0,29.0,36.0,43.0,65.0,9.485509
gender,20000,2,Male,10045,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income,20000.0,NaN,NaN,NaN,46527.615,18000.0,30900.0,41900.0,56900.0,180000.0,21968.90811
employment_status,20000,4,Employed,12430,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_score,20000.0,NaN,NaN,NaN,681.6584,438.0,644.0,682.0,719.0,850.0,55.685219
loan_amount,20000.0,NaN,NaN,NaN,8865.125,2000.0,5100.0,7600.0,11200.0,30000.0,5192.31338
loan_term_months,20000.0,NaN,NaN,NaN,32.1324,12.0,24.0,36.0,36.0,60.0,14.013775
application_started,20000.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,0.0


In [5]:
quality_report(df)

{'rows': 20000,
 'columns': 16,
 'duplicate_user_ids': 0,
 'missing_values': 16824,
 'group_counts': {'control': 10046, 'treatment': 9954},
 'invalid_binary_values': {'application_started': [],
  'application_completed': [],
  'previous_customer': [],
  'loan_approved': []},
 'started_not_one': 0,
 'completed_without_start': 0,
 'approved_without_completion': 0}

In [6]:
validate_data(df)
print('Data validation PASSED')

Data validation PASSED


## 2. Experiment balance
Random assignment should produce reasonably similar group sizes and broadly comparable baseline characteristics.


In [7]:
display(group_summary(df))
baseline=df.groupby('experiment_group').agg(users=('user_id','count'),avg_age=('age','mean'),avg_income=('income','mean'),avg_credit_score=('credit_score','mean'),avg_loan_amount=('loan_amount','mean')).round(2)
display(baseline)

,experiment_group,users,completed,completion_rate,approved,approval_rate,avg_completion_time
0,control,10046,1446,0.143938,530,0.052757,7.660221
1,treatment,9954,1730,0.173799,626,0.062889,6.931642


,users,avg_age,avg_income,avg_credit_score,avg_loan_amount
experiment_group,,,,,
control,10046,36.40,46356.87,682.04,8823.27
treatment,9954,36.45,46699.94,681.27,8907.36


## 3. Hypotheses and primary metric
H0: p_treatment = p_control. H1: p_treatment != p_control. We use a two-sided alpha of 0.05. The primary metric is application completion rate.


In [8]:
r=experiment_result(df)
print(f"Control conversion: {r['control_rate']:.2%}")
print(f"Treatment conversion: {r['treatment_rate']:.2%}")
print(f"Absolute lift: {r['absolute_lift']:.2%}")
print(f"Relative lift: {r['relative_lift']:.2%}")


Control conversion: 14.39%
Treatment conversion: 17.38%
Absolute lift: 2.99%
Relative lift: 20.75%


## 4. Statistical significance
The primary outcome is binary, so we use a two-proportion z-test.


In [ ]:
print(f"Z-statistic: {r['z_statistic']:.4f}")
print(f"P-value: {r['p_value']:.6g}")
print(f"Significant at alpha=0.05: {r['significant_at_alpha_0_05']}")

Z-statistic: 5.7772
P-value: 7.59475e-09
Significant at alpha=0.05: True


## 5. Confidence intervals
The treatment effect is estimated with a 95% confidence interval.


In [ ]:
print('Control 95% CI:', [f'{x:.2%}' for x in r['control_ci']])
print('Treatment 95% CI:', [f'{x:.2%}' for x in r['treatment_ci']])
print('Treatment-control 95% CI:', [f'{x:.2%}' for x in r['difference_ci']])

Control 95% CI: ['13.71%', '15.08%']
Treatment 95% CI: ['16.64%', '18.12%']
Treatment-control 95% CI: ['1.97%', '4.00%']


## 6. Secondary metrics


In [12]:
secondary=df.groupby('experiment_group').agg(users=('user_id','count'),avg_completion_time=('application_time_minutes','mean'),loan_approval_rate=('loan_approved','mean')).round(3)
display(secondary)

,users,avg_completion_time,loan_approval_rate
experiment_group,,,
control,10046,7.660,0.053
treatment,9954,6.932,0.063


## 7. Exploratory segment analysis
Segment results are exploratory and should not replace the primary experiment test.


In [13]:
display(segment_conversion(df,'device_type'))
display(segment_conversion(df,'employment_status'))


,device_type,experiment_group,users,completed,conversion_rate
0,Mobile,control,6254,896,0.143268
1,Mobile,treatment,6119,1057,0.172741
2,Tablet,control,603,76,0.126036
3,Tablet,treatment,601,85,0.141431
4,Web,control,3189,474,0.148636
5,Web,treatment,3234,588,0.181818


,employment_status,experiment_group,users,completed,conversion_rate
0,Contract,control,1276,162,0.126959
1,Contract,treatment,1312,237,0.180640
2,Employed,control,6222,911,0.146416
3,Employed,treatment,6208,1113,0.179285
4,Self-employed,control,2046,290,0.141740
5,Self-employed,treatment,1950,318,0.163077
6,Unemployed,control,502,83,0.165339
7,Unemployed,treatment,484,62,0.128099


## 8. Final interpretation checklist
1. What happened to conversion?
2. What are absolute and relative lift?
3. Is the difference statistically significant?
4. What is the confidence interval?
5. What happened to secondary metrics?
6. Are segment patterns consistent?
7. Is the effect practically meaningful?
8. What are the limitations of synthetic data?
